In [1]:
import sksurv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
import pandas as pd
import numpy as np
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import os
import kagglehub
from dataclasses import asdict
from sksurv.util import Surv

from Utils.Config import Config, TrialResult
from Utils.utils import (
    set_seed,
    make_surv_y,
    make_oof_predictions,
    create_features,
    load_config_yaml,
    make_corr_matrix,
    compute_hybrid_score,
    find_ensemble_model,
    HORIZONS
)

In [2]:
storage = JournalStorage(
    JournalFileBackend("gbsa_journal.log")
)

study = optuna.load_study(
    study_name="gbsa_survival",
    storage=storage,
)

In [3]:
best_trial = study.best_trial
best_trial_num = best_trial.number
print("Best trial:")
print(f"  Number: {best_trial.number}")
print(f"  Value: {best_trial.value}")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Number: 106
  Value: 0.9629674137093398
  Params: 
    n_estimators: 417
    learning_rate: 0.11063731185373184
    subsample: 0.5905037758154414
    max_depth: 6
    max_features: sqrt
    max_leaf_nodes: 31
    min_samples_split: 4
    min_samples_leaf: 8
    min_impurity_decrease: 0.02904783931441647
    ccp_alpha: 0.004992133979210339
    dropout_rate: 0.0007471276215608402
    validation_fraction: 0.1897124693474042
    n_iter_no_change: None
    tol: 0.0004912572712305443


In [4]:
trial_dir = os.path.join("Trials", "GBSA")
best_trial_path = os.path.join(trial_dir, f'trial_{best_trial.number}', 'config.yaml')
print(f"Best trial config path: {best_trial_path}")
print(f"Best trial config exists: {os.path.exists(best_trial_path)}")

Best trial config path: Trials/GBSA/trial_106/config.yaml
Best trial config exists: True


In [5]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')

In [6]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_processed = create_features(train_df)
test_processed = create_features(test_df)

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

event_horizon = np.array(HORIZONS).copy()
event_horizon[-1] = min(event_horizon[-1], train_processed['time_to_hit_hours'].max() - 1e-6)

print("Event horizons:", event_horizon)

Event horizons: [12.         24.         48.         66.99447313]


In [7]:
config = load_config_yaml(best_trial_path)
print("Loaded config:")
print(config)

Loaded config:
Config(seed=42, cv_n_splits=5, cv_n_repeats=10, gbsa_config=GBSAConfig(loss='coxph', n_estimators=417, learning_rate=0.11063731185373184, subsample=0.5905037758154414, max_depth=6, max_features='sqrt', max_leaf_nodes=31, min_samples_split=4, min_samples_leaf=8, min_weight_fraction_leaf=0.0, min_impurity_decrease=0.02904783931441647, criterion='friedman_mse', ccp_alpha=0.004992133979210339, dropout_rate=0.0007471276215608402, validation_fraction=0.1897124693474042, n_iter_no_change=None, tol=0.0004912572712305443, random_state=42, warm_start=False, verbose=0), preprocessing_config=PreprocessingConfig(eps=1e-06, min_speed=0.01, max_hours=9999.0))


In [8]:
seed = config.seed
set_seed(seed)
model_params = config.gbsa_config

n_splits = config.cv_n_splits
n_repeats = config.cv_n_repeats

print(n_splits, n_repeats)

5 10


# Model Ensemble (differnt Config)

In [9]:
from Utils.utils import get_top_trial_oofs
result = get_top_trial_oofs(
    study=study,
    data=train_processed,
    horizons=event_horizon,
    top_ratio=0.3,
    seed=seed,
    n_splits=n_splits,
    n_repeats=n_repeats,
    out_dir=os.path.join("Top OOF", "GBSA")
)

result_item_list = list(result.values())
print(result_item_list)

print(result_item_list[0].keys())
print(result_item_list[0])

Saving OOF predictions for top 90 trials to Top OOF/GBSA...
Best trial value: 0.9629674137093398
Best Tral Number: 106
[{'value': 0.9629674137093398, 'value_std': 0.0013867783690788287, 'trial_id': 106, 'oof_risk': array([-0.95422292,  2.80563705,  3.73880235, -1.55702051, -1.26366121,
       -1.47569768,  2.40444177, -1.13313123, -1.39665391,  1.65003136,
       -1.51454957,  2.85668923,  1.85835523, -1.52302872, -1.53202476,
       -1.48201131,  2.50957054,  2.44415547,  4.66897514, -1.53316197,
       -1.12338525, -1.52112214, -1.5117804 , -1.48538701,  4.52213161,
       -1.47907024,  4.05278267,  2.87509405, -1.5084563 , -1.53848902,
        4.41753685,  3.56094777,  2.19813216, -1.52441536, -1.51625807,
       -1.50933459,  2.62544227,  2.23949006, -1.50414651, -1.39688308,
       -1.4878827 , -1.17103745,  4.26306884,  2.89239985,  2.45192505,
       -1.33964277, -1.5050198 ,  2.01218285, -1.44109376, -1.52850725,
       -1.07047606, -1.50374923, -1.47162335,  1.88990033, -1.503

In [10]:
result_item_list[0]['oof_surv']

array([[9.68416809e-01, 9.26177791e-01, 9.05163095e-01, 7.94469377e-01],
       [2.52884653e-01, 5.21538467e-02, 2.15853070e-02, 7.34015391e-04],
       [4.06431023e-02, 1.21119798e-03, 1.69512495e-04, 1.39199429e-05],
       [9.82573108e-01, 9.54916422e-01, 9.43073910e-01, 8.88311518e-01],
       [9.77584078e-01, 9.43282641e-01, 9.28751148e-01, 8.61628624e-01],
       [9.82640595e-01, 9.60052780e-01, 9.48937880e-01, 8.92270923e-01],
       [4.28232135e-01, 1.47546551e-01, 8.12557160e-02, 1.30410885e-02],
       [9.75192485e-01, 9.40309450e-01, 9.24699203e-01, 8.52186139e-01],
       [9.81422235e-01, 9.53927262e-01, 9.40793132e-01, 8.82820457e-01],
       [6.52386967e-01, 3.58447587e-01, 2.68992480e-01, 9.54467661e-02],
       [9.82797318e-01, 9.56331667e-01, 9.45905186e-01, 8.96455627e-01],
       [2.74665249e-01, 5.20379104e-02, 2.29271972e-02, 7.01100692e-04],
       [6.17430856e-01, 2.91797770e-01, 2.08671317e-01, 5.98049786e-02],
       [9.83609423e-01, 9.59139210e-01, 9.46110418e

# Select Ensemble Model

- 대회에 쓰이는 metric을 직접 최적화 하는 방식으로 진행

## 제약조건

1. 전체 trial중 30% (90개)
2. diversity를 어느정도 갖춘 core model [106,152,236,243]으로 시작
3. 이미 선택된 모델들과 corr이 0.995 이상으로 강하다면 skip
4. improve가 너무 적게 상승한다면 noise로 보고 skip
5. max model은 10개 정도 진행

In [11]:
is_sorted = True
prev_value = 10000.0

for i, item in enumerate(result_item_list):
    value = item['value']
    
    if value > prev_value:
        is_sorted = False
        break

    prev_value = value
    
print(is_sorted)

True


## Config

In [12]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')
seed = 42
set_seed(seed)

train_df = pd.read_csv(train_path)
train_processed = create_features(train_df)

print("Metadata path: ", metadata_path)
print("Train path: ", train_path)
print("Test path: ", test_path)
print()

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

Metadata path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv
Train path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/train.csv
Test path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/test.csv



In [ ]:
# init_model_list = [106,152,236,243]
init_model_list = None
max_pair_corr = 0.995
max_ensemble_corr = 0.990
min_improvement_score = 0.0003
max_model_num = 10

In [14]:
ensemble_result = find_ensemble_model(
    oof_result=result,
    label=y_full,
    max_ensemble_corr=max_ensemble_corr,
    max_pair_corr=max_pair_corr,
    min_imporvement_score=min_improvement_score,
    max_model_num=max_model_num,
    init_model_list=init_model_list,
    horizons=HORIZONS,
    verbose=True,
)

===== Initial Ensemble =====
Initial models: [106, 152, 236, 243]
Initial hybrid score: 0.963357
Allow duplicate: False
Max select per model: 3

Current ensemble size: 4
Current hybrid score: 0.963357
[Trial 200] skip (pair corr=0.99803 > 0.995)
[Trial 105] skip (pair corr=0.99924 > 0.995)
[Trial 220] skip (pair corr=0.99576 > 0.995)
[Trial 297] skip (pair corr=0.99889 > 0.995)
[Trial 163] skip (pair corr=0.99923 > 0.995)
[Trial 216] skip (pair corr=0.99978 > 0.995)
[Trial 179] skip (pair corr=0.99796 > 0.995)
[Trial 134] skip (pair corr=0.99911 > 0.995)
[Trial 238] skip (pair corr=0.99889 > 0.995)
[Trial 130] skip (pair corr=0.99762 > 0.995)
[Trial 284] skip (pair corr=0.99786 > 0.995)
[Trial 80] skip (pair corr=0.99689 > 0.995)
[Trial 296] skip (pair corr=0.99932 > 0.995)
[Trial 257] skip (pair corr=0.99651 > 0.995)
[Trial 189] skip (pair corr=0.99508 > 0.995)
[Trial 83] skip (pair corr=0.99666 > 0.995)
[Trial 198] skip (pair corr=0.99706 > 0.995)
[Trial 100] skip (ensemble corr=0.99